# Clustering Hybride — Complement Zipline + Micro-Hubs DHG

Auteur : Mathieu LE FAOU  
Date : Mars 2026  
Approche : Modelisation d'un reseau complementaire aux 6 hubs Zipline existants au Ghana. Les zones deja couvertes par Zipline (rayon 80 km) sont identifiees, puis un K-Means incremental positionne des micro-hubs (rayon 30 km) pour atteindre 98% de couverture des facilities de sante.

---

Problematique : Zipline opere 6 centres au Ghana avec un rayon de 80 km. Quels micro-hubs complementaires permettraient de couvrir les zones que Zipline ne dessert pas, en minimisant le nombre de hubs additionnels ?

Valeur ajoutee : cette approche repond au critere d'evaluation Jedha n 9 (performance vs existant) en quantifiant l'apport sur un reseau deja partiellement deploye.

---

### Donnees de depart
- `ghana_health_eda_final.csv` : facilities de sante geocodees (~2 400 points)
- `ghana_villages_eda_final.csv` : villages avec facility de reference et population


---
## 1. Configuration et parametres

Les constantes metier sont fixees par le cahier des charges du projet :
- `MAX_RADIUS_KM = 80` : autonomie maximale des drones Zipline (AR 160 km / 2)
- `RADIUS_MICRO_KM = 30` : rayon opérationnel reduit des micro-hubs DHG, adapte aux zones denses mais non urbaines
- `TARGET_PCT = 0.98` : objectif de couverture cible (98% des facilities)



In [ ]:
import warnings
from pathlib import Path

import folium
from folium import plugins
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans


warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", "{:.2f}".format)

print("Imports OK")

In [ ]:
# --- Constantes metier ---
MAX_RADIUS_KM: float = 80.0        # rayon Zipline / contrainte Haversine projet
RADIUS_MICRO_KM: float = 30.0      # rayon micro-hub DHG complementaire
TARGET_PCT: float = 0.98           # objectif couverture nationale



# Hubs Zipline operationnels au Ghana (coordonnees GPS publiques)
ZIPLINE_HUBS: list[dict] = [
    {"name": "Omenako",      "lat": 6.2027,  "lon": -0.4462},
    {"name": "Mpanya",       "lat": 7.0545,  "lon": -1.3855},
    {"name": "Vobsi",        "lat": 10.3470, "lon": -0.8035},
    {"name": "Sefwi Wiawso", "lat": 6.1820,  "lon": -2.4845},
    {"name": "Anum",         "lat": 6.4885,  "lon":  0.1625},
    {"name": "Kete Krachi",  "lat": 7.7981,  "lon": -0.0125},
]

# Chemins (notebooks/contributions_modeling/ -- 2 niveaux pour remonter a la racine)
DATA_DIR = Path("../..")
RAW_DIR = DATA_DIR / "data" / "Final Group"
PROCESSED_DIR = DATA_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Rayon Zipline   : {MAX_RADIUS_KM} km")
print(f"Rayon micro-hub : {RADIUS_MICRO_KM} km")
print(f"Cible couverture: {TARGET_PCT*100:.0f}%")

---
## 2. Fonctions de calcul

La distance Haversine est obligatoire dans ce projet (contrainte metier) : elle tient compte de la courbure terrestre, ce qu'une distance euclidienne sur coordonnées brutes ne ferait pas. L'erreur serait de l'ordre de 0,3% a l'echelle du Ghana, mais le cahier des charges l'impose explicitement.

La version vectorisee `haversine_vec` calcule la distance entre un point scalaire et un tableau de points en une seule operation NumPy, ce qui est indispensable pour la boucle d'optimisation K-Means.

In [ ]:
def haversine_vec(lat1: float, lon1: float, lats2: np.ndarray, lons2: np.ndarray) -> np.ndarray:
    """Distance Haversine (km) entre un point scalaire et un tableau de points."""
    R = 6371.0
    la1, lo1 = np.radians(lat1), np.radians(lon1)
    la2 = np.radians(lats2)
    lo2 = np.radians(lons2)
    dlat = la2 - la1
    dlon = lo2 - lo1
    a = np.sin(dlat / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))


def haversine_matrix(
    lats1: np.ndarray, lons1: np.ndarray,
    lats2: np.ndarray, lons2: np.ndarray,
) -> np.ndarray:
    """Matrice de distances Haversine vectorisee (km) -- broadcasting NumPy."""
    R = 6371.0
    la1 = np.radians(lats1)
    la2 = np.radians(lats2)
    dlat = la2 - la1
    dlon = np.radians(lons2) - np.radians(lons1)
    a = np.sin(dlat / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))


# Verification : Accra -> Kumasi ~ 270 km
test = haversine_vec(5.6037, -0.1870, np.array([6.6885]), np.array([-1.6244]))
print(f"Test Haversine Accra -> Kumasi : {test[0]:.1f} km")

---
## 3. Chargement des donnees

Deux fichiers produits par le pipeline ETL sont utilises :
- ghana_health_eda_final.csv : facilities de sante geocodees et deduplicatees (~2 400 points)
- ghana_villages_eda_final.csv : villages avec leur facility de reference et leur population

Le clustering Zipline s'applique aux **facilities** (ce sont les points de livraison, pas les villages). La population desservie est calculee a partir des villages rattaches a chaque facility.

In [ ]:
facilities = pd.read_csv(RAW_DIR / "ghana_health_eda_final.csv")
villages = pd.read_csv(RAW_DIR / "ghana_villages_eda_final.csv")

facilities = facilities.dropna(subset=["Latitude", "Longitude"]).reset_index(drop=True)
villages = villages.dropna(subset=["Latitude", "Longitude"]).reset_index(drop=True)

print(f"Facilities chargees : {len(facilities):,} lignes, {facilities.shape[1]} colonnes")
print(f"Villages charges    : {len(villages):,} lignes, {villages.shape[1]} colonnes")

In [ ]:
facilities.head(3)

Enrichissement : la colonne `Pop_desservie` n'existe pas dans le fichier brut. Elle est calculee en aggreant la population des villages dont la facility est la plus proche (`Nearest_Facility_Name`). Cette ponderation permettra, dans les calculs financiers, d'estimer l'impact humain reel de chaque hub.

In [ ]:
pop_by_fac = (
    villages
    .groupby("Nearest_Facility_Name")["Population"]
    .sum()
    .reset_index()
    .rename(columns={"Population": "Pop_desservie"})
)

facilities = facilities.merge(
    pop_by_fac,
    left_on="Facility_Name",
    right_on="Nearest_Facility_Name",
    how="left",
).drop(columns=["Nearest_Facility_Name"])
facilities["Pop_desservie"] = facilities["Pop_desservie"].fillna(0).astype(int)

print(f"Facilities avec population renseignee : {(facilities['Pop_desservie'] > 0).sum()} / {len(facilities)}")
print(f"Population totale couverte            : {facilities['Pop_desservie'].sum():,}")

---
## 4. Couverture initiale Zipline

On calcule la distance Haversine entre chaque facility et les 6 hubs Zipline. Une facility est consideree **couverte** si elle se trouve dans un rayon de 80 km d'au moins un hub.

Cette etape mesure la **couverture de base** avant tout déploiement DHG. C'est le point de depart de l'analyse differentielle — le delta entre cette couverture initiale et les 98% cibles definit le travail a effectuer par les micro-hubs DHG.

In [ ]:
df = pd.read_csv(RAW_DIR / 'ghana_health_eda_final.csv')

# --- 1. DÉFINITION DES HUBS ZIPLINE (Source Officielle) ---
zipline_hubs_data = [
    {"name": "Omenako", "lat": 6.1158, "lon": -0.3894},
    {"name": "Mpanya", "lat": 6.8906, "lon": -1.4552},
    {"name": "Vobsi", "lat": 10.3340, "lon": -0.8143},
    {"name": "Sefwi Wiawso", "lat": 6.2085, "lon": -2.4839},
    {"name": "Anum", "lat": 6.4716, "lon": 0.1614},
    {"name": "Kete Krachi", "lat": 7.7811, "lon": -0.0152}
]

# Conversion en Array Numpy pour le calcul rapide
hubs_coords = np.array([[h['lat'], h['lon']] for h in zipline_hubs_data])

# --- 2. CALCUL DE LA DISTANCE POUR CHAQUE CENTRE ---
# On récupère toutes les coordonnées de votre fichier df_final
all_coords = df[['Latitude', 'Longitude']].values

# Calcul de la distance minimale de chaque centre vers le hub Zipline le plus proche
# cdist calcule toutes les combinaisons, on prend le min, puis on convertit en km
distances_to_zipline = np.min(
    np.sqrt(np.sum((all_coords[:, None, :] - hubs_coords[None, :, :]) ** 2, axis=2)),
    axis=1
) * 111

# --- 3. CRÉATION DU DF_UNCOVERED (Le "Gap") ---
# Rayon d'action standard de Zipline = 80km
df['Distance_to_Zipline'] = distances_to_zipline
df_uncovered = df[df['Distance_to_Zipline'] > 80].copy()

# --- 4. VÉRIFICATION DES STATISTIQUES ---
n_total = len(df)
n_uncovered = len(df_uncovered)
couverture_actuelle = ((n_total - n_uncovered) / n_total) * 100

print(f"--- DIAGNOSTIC DU RÉSEAU ---")
print(f"Total établissements : {n_total}")
print(f"Établissements isolés (>80km) : {n_uncovered}")
print(f"Couverture Zipline réelle : {couverture_actuelle:.2f}%")

Interpretation : le taux de couverture Zipline illustre l'etendue du reseau existant. Les facilities hors portee constituent le segment prioritaire pour MASA : ce sont elles qui beneficieront des micro-hubs complementaires. La distribution des distances au graphique suivant permet de situer les zones les plus eloignees.

---
## 5. Optimisation K-Means incrementale/MLCP 

Choix de la projection : le clustering est realise directement sur les coordonnées GPS. A l'echelle du Ghana (bande 5-11 degres N), la distorsion introduite par cet espace pseudo-euclidien reste inferieure a 1% pour le positionnement des centroides. Les distances reelles sont toujours calculees en Haversine pour verifier la couverture.

Critere d'arret : couverture totale (Zipline + DHG) >= 98%, ou K >= 50 comme garde-fou.

**A: OPTIMISATION K-MEANS :**

In [ ]:
# Methode KMeans pour trouver les micro-hubs additionnels

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist

# --- CONFIGURATION ---
TARGET_COVERAGE = 98.0  
RADIUS_KM = 30         
COST_PER_HUB = 50100   

# --- PRÉPARATION DES DONNÉES ---
# Supposons que raw_df est votre dataset complet (100%)
# et df_uncovered sont les 20% restants (hors portée Zipline)
n_total = len(df)
n_deja_couverts = n_total - len(df_uncovered)

coords_hors_portee = df_uncovered[['Latitude', 'Longitude']].values

# Initialisation de la couverture GLOBALE
current_global_coverage = (n_deja_couverts / n_total) * 100
k = 1
results = []

print(f"Couverture de base (Zipline) : {current_global_coverage:.2f}%")
print(f"Objectif : atteindre {TARGET_COVERAGE}% de couverture nationale...")

# --- BOUCLE D'OPTIMISATION ---
while current_global_coverage < TARGET_COVERAGE:
    # On entraîne le KMeans uniquement sur les points non couverts
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42).fit(coords_hors_portee)
    centers = kmeans.cluster_centers_
    
    # On calcule combien de nouveaux points sont couverts par ces K micro-hubs
    distances = cdist(coords_hors_portee, centers) * 111
    min_dist = np.min(distances, axis=1)
    n_nouveaux_couverts = np.sum(min_dist <= RADIUS_KM)
    
    # LA FORMULE CRUCIALE : (Anciens + Nouveaux) / Total National
    current_global_coverage = ((n_deja_couverts + n_nouveaux_couverts) / n_total) * 100
    
    results.append({
        'k': k,
        'global_coverage': current_global_coverage,
        'cost': k * COST_PER_HUB
    })
    
    if k % 5 == 0:
        print(f"Hubs additionnels : {k} | Couverture totale : {current_global_coverage:.2f}%")
    
    # Sécurités
    if k >= len(df_uncovered): break
    if k > 300: break 
        
    if current_global_coverage < TARGET_COVERAGE:
        k += 1

# --- RÉSULTATS ---
if results:
    res = results[-1]
    print("\n" + "="*40)
    print(f"ANALYSE DE COMPLÉMENTARITÉ TERMINEE")
    print(f"Nombre de Micro-Hubs à ajouter : {res['k']}")
    print(f"Couverture finale atteinte : {res['global_coverage']:.2f}%")
    print(f"Budget d'extension requis : ${res['cost']:,.2f}")
    print("="*40)

Interpretation : le nombre K determine que 26 micro-hubs MASA sont nécessaires pour atteindre 98% de couverture nationale.

B : Test de la méthode MCLP (Optimisation Intégrée) :

In [ ]:
# METHODE MCLP INTÉGRÉE (Optimisation directe de la couverture GLOBALE)

import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist

# --- CONFIGURATION ---
TARGET_GLOBAL_COVERAGE = 98.0  # Objectif national
RADIUS_KM = 30
COST_PER_HUB = 50100

# --- DONNÉES DE BASE ---
# n_total_ghana : nombre total de centres dans votre dataset initial
# points_uncovered : les 20% de centres hors portée de Zipline
n_total_ghana = len(df) 
n_deja_couverts_zipline = n_total_ghana - len(df_uncovered)
points_orphelins = df_uncovered[['Latitude', 'Longitude']].values

def run_mclp_integrated(points_to_cover, n_total, n_base_covered, radius_km, target_pct):
    covered_in_session = set()
    selected_hubs = []
    
    # On utilise les points orphelins comme emplacements de hubs potentiels
    potential_hubs = points_to_cover.copy()
    dist_matrix = cdist(potential_hubs, points_to_cover) * 111
    
    # Calcul initial de la couverture globale
    current_global_cov = (n_base_covered / n_total) * 100
    
    print(f"Démarrage : {current_global_cov:.2f}% de couverture (Zipline)")
    
    while current_global_cov < target_pct:
        best_gain = -1
        best_hub_idx = -1
        best_new_points = set()
        
        # Recherche du hub avec le meilleur impact sur la couverture GLOBALE
        for i in range(len(potential_hubs)):
            in_range = set(np.where(dist_matrix[i] <= radius_km)[0])
            # On ne compte que les points qui ne sont PAS ENCORE couverts par les nouveaux hubs
            gain_points = in_range - covered_in_session
            gain = len(gain_points)
            
            if gain > best_gain:
                best_gain = gain
                best_hub_idx = i
                best_new_points = gain_points
        
        # Stop si plus aucun gain possible
        if best_gain <= 0:
            print("Alerte : Impossible d'atteindre l'objectif, points trop dispersés.")
            break
            
        # Mise à jour
        selected_hubs.append(potential_hubs[best_hub_idx])
        covered_in_session.update(best_new_points)
        
        # Nouveau calcul de couverture sur l'échelle nationale
        current_global_cov = ((n_base_covered + len(covered_in_session)) / n_total) * 100
        
        if len(selected_hubs) % 5 == 0:
            print(f"Micro-Hubs ajoutés: {len(selected_hubs)} | Couverture Totale: {current_global_cov:.2f}%")

    return np.array(selected_hubs), current_global_cov

# --- EXÉCUTION ---
hubs_optimaux, couverture_nationale = run_mclp_integrated(
    points_orphelins, 
    n_total_ghana, 
    n_deja_couverts_zipline, 
    RADIUS_KM, 
    TARGET_GLOBAL_COVERAGE
)


print("-" * 30)
print(f"RÉSULTATS FINAUX (SYNERGIE ZIPLINE + MICRO-HUBS) :")
print(f"Nombre de Micro-Hubs additionnels : {len(hubs_optimaux)}")
print(f"Couverture nationale atteinte : {couverture_nationale:.2f}%")
print(f"Coût de l'extension : ${len(hubs_optimaux) * COST_PER_HUB:,.2f}")
print("-" * 30)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

# --- 1. FONCTION DE CALCUL MCLP SEULE ---
def generate_mclp_data():
    pts = points_orphelins  
    n_base = n_deja_couverts_zipline
    n_total = n_total_ghana
    
    hist_mclp = []
    covered_indices_mclp = set()
    dist_matrix = cdist(pts, pts) * 111
    
    current_cov = (n_base / n_total) * 100
    k = 0
    
    while current_cov < TARGET_GLOBAL_COVERAGE:
        k += 1
        best_gain = -1
        best_idx = -1
        
        for i in range(len(pts)):
            # On regarde combien de nouveaux points ce hub potentiel couvre
            in_range = set(np.where(dist_matrix[i] <= RADIUS_KM)[0])
            gain = len(in_range - covered_indices_mclp)
            
            if gain > best_gain:
                best_gain = gain
                best_idx = i
        
        if best_gain <= 0: 
            print("Note : Impossible de gagner plus de couverture avec le rayon actuel.")
            break
        
        # Mise à jour de la couverture
        covered_indices_mclp.update(set(np.where(dist_matrix[best_idx] <= RADIUS_KM)[0]))
        current_cov = ((n_base + len(covered_indices_mclp)) / n_total) * 100
        hist_mclp.append((k, current_cov))
        
        if k > 500: break # Sécurité
    
    return hist_mclp

# --- 2. GÉNÉRATION DES RÉSULTATS ---
h_mclp = generate_mclp_data()

# --- 3. AFFICHAGE DU GRAPHIQUE ---
plt.figure(figsize=(10, 6))
plt.plot([x[0] for x in h_mclp], [x[1] for x in h_mclp], 'g-', label='Optimisation MCLP', linewidth=2.5)
plt.axhline(y=TARGET_GLOBAL_COVERAGE, color='red', linestyle=':', label=f'Objectif {TARGET_GLOBAL_COVERAGE}%')
plt.axhline(y=(n_deja_couverts_zipline/n_total_ghana)*100, color='black', linestyle='-', alpha=0.2, label='Base Zipline')

plt.title('Stratégie d\'Expansion Optimale (MCLP)')
plt.xlabel('Nombre de Micro-Hubs Additionnels ($K$)')
plt.ylabel('Couverture Nationale Totale (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# --- 4. PRINT DES RÉSULTATS DÉTAILLÉS ---
def print_mclp_results(h_mclp):
    print("\n" + "="*50)
    print("RÉSULTATS DE L'OPTIMISATION MCLP")
    print("="*50)
    
    final_step = h_mclp[-1]
    total_cost = final_step[0] * COST_PER_HUB
    
    print(f"{'Hubs Micro-Réseau nécessaires':<30} : {final_step[0]}")
    print(f"{'Couverture Nationale Finale':<30} : {final_step[1]:.2f}%")
    print(f"{'Investissement Total estimé':<30} : ${total_cost:,}")
    print("-" * 50)
    
    print("PROGRESSION DES PALIERS :")
    for pct in [90, 95, 98]:
        step = next((x for x in h_mclp if x[1] >= pct), None)
        if step:
            print(f"-> Palier {pct}% : {step[0]} hubs | Coût : ${step[0]*COST_PER_HUB:,}")
    print("="*50)

print_mclp_results(h_mclp)



**Conclusion du comparatif des modeles Kmeans et MCLP :** La comparaison de ces deux modèles montrent que la méthode MCLP montre les meilleurs résultats avec l'implantation de 25 Hubs pour une couverture total de 98.13 %, contre 26 hubs pour la Méthode K-means avec une couverture de 98,01 %


## 6. Calcul de l'efficacité de l'investissement "ROI" 



In [ ]:
# --- CALCUL DU ROI ---
results_roi = []
total_points = len(points_orphelins)

# On analyse l'historique du MCLP (le plus performant)
for n_hubs, coverage in h_mclp:
    total_cost = n_hubs * COST_PER_HUB
    points_covered = int((coverage / 100) * total_points)
    
    # Coût moyen par centre de santé couvert
    cost_per_center = total_cost / points_covered if points_covered > 0 else 0
    
    results_roi.append({
        'hubs': n_hubs,
        'coverage': coverage,
        'total_cost': total_cost,
        'cost_per_center': cost_per_center
    })

roi_df = pd.DataFrame(results_roi)

# --- VISUALISATION DU COÛT MARGINAL ---
plt.figure(figsize=(10, 5))
plt.plot(roi_df['coverage'], roi_df['cost_per_center'], color='purple', marker='x')
plt.title('Efficacité de l\'Investissement (Coût par Centre Couvert)')
plt.xlabel('Couverture Totale (%)')
plt.ylabel('Coût moyen par centre ($)')
plt.grid(True, alpha=0.3)
plt.show()

# --- RÉSUMÉ EXÉCUTIF ---
final_roi = roi_df.iloc[-1]
print(f"--- RAPPORT DE RENTABILITÉ (Objectif {TARGET_COVERAGE}%) ---")
print(f"Nombre total de hubs : {int(final_roi['hubs'])}")
print(f"Investissement Total : ${final_roi['total_cost']:,.2f}")
print(f"Coût moyen par centre de santé sécurisé : ${final_roi['cost_per_center']:,.2f}")

---
## 7. Visualisation — Master Plan Hybride

La carte represente les 3 couches du reseau :
1. Points de chaleur : densité de population sur le territoire
2. **Hubs Zipline** (icone avion, cercle sombre 80 km) : couverture existante
3. **Micro-hubs MASA** (icone eclair, cercle bleu 30 km) : solution proposée


In [ ]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import Fullscreen, HeatMap
from geopy.distance import geodesic
from sklearn.cluster import KMeans

# ==========================================
# 1. CHARGEMENT ET NETTOYAGE (SANTÉ + VILLAGES)
# ==========================================
print("Chargement des données...")
try:
    df = pd.read_csv(RAW_DIR / 'ghana_health_eda_final.csv')
    df_villages = pd.read_csv(RAW_DIR / 'ghana_villages_eda_final.csv') # Ton 2ème dataset
except FileNotFoundError as e:
    print(f"Erreur : {e}")
    exit()

# --- Nettoyage Santé ---
df = df.dropna(subset=['Latitude', 'Longitude'])
mask_shift = df['Latitude'].astype(str).str.contains('[a-zA-Z]', na=False)
if mask_shift.any():
    cols = df.columns.tolist()
    idx_type, idx_lat = cols.index('Facility_Type'), cols.index('Latitude')
    df.loc[mask_shift, 'Facility_Name'] = df.loc[mask_shift, 'Facility_Name'].astype(str) + ", " + df.loc[mask_shift, 'Facility_Type'].astype(str)
    df.loc[mask_shift, cols[idx_type:-1]] = df.loc[mask_shift, cols[idx_lat:]].values
    df.loc[mask_shift, cols[-1]] = np.nan

df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df = df.dropna(subset=['Latitude', 'Longitude'])

# Catégorisation pour la légende
def normalize_cat(ftype):
    ftype = str(ftype).lower()
    if 'hospital' in ftype: return 'Hospital'
    if 'clinic' in ftype: return 'Clinic'
    if 'chps' in ftype: return 'CHPS'
    return 'Other'
df['Facility_Category'] = df['Facility_Type'].apply(normalize_cat)

# --- Nettoyage Villages ---
df_villages = df_villages.dropna(subset=['Latitude', 'Longitude'])

# ==========================================
# 2. CALCULS DE COUVERTURE ET K-MEANS (K=25)
# ==========================================
ZIPLINE_HUBS = [
    {"name": "Omenako", "loc": [6.2027, -0.4462]}, {"name": "Mpanya", "loc": [7.0545, -1.3855]},
    {"name": "Vobsi", "loc": [10.3470, -0.8035]}, {"name": "Sefwi Wiawso", "loc": [6.1820, -2.4845]},
    {"name": "Anum", "loc": [6.4885, 0.1625]}, {"name": "Kete Krachi", "loc": [7.7981, -0.0125]}
]

def get_min_dist(row, hubs):
    p = (row['Latitude'], row['Longitude'])
    return min([geodesic(p, h['loc'] if isinstance(h, dict) else h).km for h in hubs])

# Couverture Initiale
df['Dist_to_Hub'] = df.apply(lambda r: get_min_dist(r, ZIPLINE_HUBS), axis=1)
df['Is_Covered'] = (df['Dist_to_Hub'] <= 80).astype(int)

# On applique les resultats de la mclp de la cellule precedente pour trouver les nouveaux hubs optimaux et calculer la couverture finale
new_hubs_coords = hubs_optimaux  # Ces coordonnées viennent de la MCLP


# Couverture Finale
df['Final_Covered'] = df.apply(lambda r: 1 if min(get_min_dist(r, ZIPLINE_HUBS), get_min_dist(r, new_hubs_coords)) <= 80 else 0, axis=1)

# Stats pour la légende
pct_init = (df['Is_Covered'].sum() / len(df)) * 100
pct_final = (df['Final_Covered'].sum() / len(df)) * 100

# ==========================================
# 3. ANALYSE FINANCIÈRE (Basée sur ton rapport)
# ==========================================
# Selon ton rapport, un hub "Standard" coûte :
CAPEX_PER_HUB = 50100  
OPEX_PER_MONTH = 36100   

total_capex_new = 25 * CAPEX_PER_HUB
total_opex_annual_new = OPEX_PER_MONTH * 12

# ==========================================
# 4. GÉNÉRATION DE LA CARTE MASTER PLAN
# ==========================================
m = folium.Map(location=[7.9, -1.0], zoom_start=7,tiles="OpenStreetMap")

# Couche 1 : Densité de Population (Heatmap villages)
fg_pop = folium.FeatureGroup(name="🔥 Densité Population (Villages)", show=False)
heat_data = [[r['Latitude'], r['Longitude']] for _, r in df_villages.iterrows()]
HeatMap(heat_data, radius=10, blur=15).add_to(fg_pop)

# Couche 2 : Centres de Santé (Couleurs pro)
fg_health = folium.FeatureGroup(name="🏥 Centres de Santé", show=True)
COLOR_MAP = {'Hospital': '#d35400', 'Clinic': '#2980b9', 'CHPS': '#27ae60', 'Other': '#7f8c8d'}
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=1.5,
        color=COLOR_MAP.get(row['Facility_Category'], 'gray'),
        fill=True, fill_opacity=0.5
    ).add_to(fg_health)

# Couche 3 : Hubs Zipline (Noir)
fg_zipline = folium.FeatureGroup(name="Existing Zipline Network")
for h in ZIPLINE_HUBS:
    folium.Marker(h['loc'], icon=folium.Icon(color='black', icon='plane', prefix='fa')).add_to(fg_zipline)
    folium.Circle(h['loc'], radius=80000, color='black', fill=True, opacity=0.05).add_to(fg_zipline)

# Couche 4 : Nouveaux Hubs DHG (Rouge)
fg_new = folium.FeatureGroup(name="🚀 MASA Optimized Extension")
for i, coords in enumerate(new_hubs_coords):
    folium.Marker(coords, icon=folium.Icon(color='red', icon='star', prefix='fa'), popup=f"Hub Optimisé {i+1}").add_to(fg_new)
    folium.Circle(coords, radius=30000, color='red', fill=True, opacity=0.1, dash_array='5,5').add_to(fg_new)

# --- LÉGENDE COMPLEXE (Santé + Finance) ---
legend_html = f'''
<div style="position: fixed; bottom: 30px; left: 30px; width: 300px; z-index:9999; background-color: white; 
     padding: 15px; border: 2px solid #2C3E50; border-radius: 10px; font-family: sans-serif; font-size: 12px;">
    <b style="font-size: 14px;">📊 MASTER PLAN MASA (K=25)</b><br><br>
    <b>Couverture Sanitaire :</b><br>
    Initiale: 73,37% | <b style="color:green;">Finale: 98,13%</b><br><br>
    <b>Légende Santé :</b><br>
    <i style="background:#d35400;width:10px;height:10px;display:inline-block"></i> Hôpital 
    <i style="background:#2980b9;width:10px;height:10px;display:inline-block"></i> Clinique<br>
    <i style="background:#27ae60;width:10px;height:10px;display:inline-block"></i> CHPS 
    <i style="background:#7f8c8d;width:10px;height:10px;display:inline-block"></i> Autre<br><br>
    <b style="color:#E74C3C;">💰 IMPACT FINANCIER (Extension) :</b><br>
    CAPEX (Installation): <b>{total_capex_new/1e6:.1f} M$</b><br>
    OPEX (Annuel): <b>{total_opex_annual_new/1e6:.2f} M$</b><br>
    <i>Modèle: {CAPEX_PER_HUB/1e6}M$ / hub (Tech Zipline)</i>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Finalisation
fg_pop.add_to(m)
fg_health.add_to(m)
fg_zipline.add_to(m)
fg_our_hubs = fg_new.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
Fullscreen().add_to(m)

m.save('Ghana_Master_Plan_Final.html')
print("✅ Carte Master Plan générée avec succès !")

---
## 8. Export

Les résultats sont exportes dans `data/processed/` pour etre consommes par l'application Streamlit et la comparaison inter-scenarios.

In [ ]:
summary_dhg.to_csv(PROCESSED_DIR / "hubs_hybrid.csv", index=False)
print(f"Hubs DHG exportes : {len(summary_dhg)} lignes -> data/processed/hubs_hybrid.csv")

df_search.to_csv(PROCESSED_DIR / "clustering_metrics_hybrid.csv", index=False)
print(f"Metriques exportees -> data/processed/clustering_metrics_hybrid.csv")

---
## Corrections proposees (a valider par l'auteur)

| # | Element | Constat | Correction proposee | Pourquoi |
|---|---|---|---|---|
| M1 | 2 jeux coords Zipline | Cell 12 utilise des coords differentes de Cell 3/25 (diff 1-17km) | Aligner sur un seul jeu | Incoherence : couverture 73.37% calculee avec un jeu, carte avec un autre |
| M2 | Pop desservie = 1.5 milliard | Bug agregation Cell 10 (Pop Ghana = 33M) | Verifier la jointure (doublons possibles) | Valeur aberrante |
| M3 | `matplotlib.pyplot` | Non installe dans l'env `drone-ghana` | Remplacer par `plotly` | Stack projet |
| M4 | Imports redondants | KMeans importe 2 fois (Cell 2 + 16) | Consolider | Proprete code |
| M5 | Sections 8-9 manquantes | Saut dans numerotation (7 vers 10) | Renumeroter | Lisibilite |
| M6 | Rayon 30km (analyse) vs 80km (carte) | Cell 25 teste <= 80 pour micro-hubs | Question : intention = 30 ou 80 ? | Le MCLP utilise 30km mais la carte verifie 80km |
| M7 | Legende hardcodee | "73,37% / 98,13%" en dur dans HTML | Rendre dynamique | Si données changent, legende fausse |
| M8 | Cell Export (derniere) | Variables `summary_dhg` et `df_search` non definies | Supprimer ou commenter | Crash garanti |
| M9 | `cdist * 111` | Approximation degre vers km | Remplacer par Haversine | Cahier des charges = Haversine obligatoire |
| M10 | plotly importe mais non utilise | `px` et `go` importes Cell 2 | Supprimer les imports inutiles | Proprete |